# PopQA Benchmark — Latent-RAG

Evaluates the V4 injector architecture on the **PopQA** public benchmark (Mallen et al., 2022, arXiv:2212.10511).

## Pipeline
1. Load PopQA and filter to low-popularity subjects (`s_pop < threshold`)
2. Run no-context evaluation — keep only examples the base model **cannot** answer
3. Save filtered dataset as `train.csv` / `test.csv` (80/20, stratified by `prop`)
4. Extract concept vectors for `obj` (the answer entity to inject)
5. Train the injector: Phase 1 (CE + BERTTune) → Phase 2 (GRPO)
6. Evaluate: **Latent-RAG** vs **Perfect RAG** (gold answer in context)

**Entity injected**: `obj` (what the model must output as the answer)  
**Category field**: `prop` (16 Wikidata relation types, e.g. `occupation`, `country`)  
**Popularity filter**: `s_pop < 500` — subjects the model is unlikely to know about

## Cell 1: Imports & Setup

In [1]:
import os
from dotenv import load_dotenv
load_dotenv()
# ── Imports & Setup ───────────────────────────────────────────────────────────
import ast, random, os, copy, time, json, re, pathlib
import torch
import torch.nn as nn
import torch.nn.functional as F
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import google.generativeai as genai

from tqdm.auto import tqdm
from dataclasses import dataclass, field
from typing import List, Dict, Optional
from typing_extensions import TypedDict
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForCausalLM

# ── Credentials: read from environment variables ───────────────────────────────
# Set these in your shell before launching Jupyter:
#   export HF_TOKEN=hf_...
#   export GEMINI_API_KEY=AIza...
HF_TOKEN       = os.getenv("HF_TOKEN")
GEMINI_API_KEY = os.getenv("GEMINI_API_KEY")

if HF_TOKEN:
    os.environ["HF_TOKEN"] = HF_TOKEN
    print("HF_TOKEN       : set")
else:
    print("WARNING: HF_TOKEN not set — HuggingFace model download may fail")

if GEMINI_API_KEY:
    print("GEMINI_API_KEY : set")
else:
    print("WARNING: GEMINI_API_KEY not set — Gemini judge cells will fail")

# ── GPU check ──────────────────────────────────────────────────────────────────
if not torch.cuda.is_available():
    raise RuntimeError("No CUDA GPU detected. Check your RunAI GPU allocation.")

print(f"\nGPU    : {torch.cuda.get_device_name(0)}")
print(f"VRAM   : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
print(f"CUDA   : {torch.version.cuda}")
print(f"PyTorch: {torch.__version__}")

/opt/conda/lib/python3.10/site-packages/google/api_core/_python_version_support.py:275: FutureWarning: You are using a Python version (3.10.13) which Google will stop supporting in new releases of google.api_core once it reaches its end of life (2026-10-04). Please upgrade to the latest Python version, or at least Python 3.11, to continue receiving updates for google.api_core past that date.
  warnings.warn(message, FutureWarning)
/opt/conda/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/tmp/ipykernel_6403/3757807031.py:9: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-a

HF_TOKEN       : set
GEMINI_API_KEY : set

GPU    : NVIDIA H100 80GB HBM3
VRAM   : 84.9 GB
CUDA   : 12.8
PyTorch: 2.10.0+cu128


## Cell 2: Configuration

In [2]:
# ── Config ────────────────────────────────────────────────────────────────────
# Results are saved to: <this notebook's directory>/PopQA/
_RESULTS_DIR = str(pathlib.Path.cwd() / "PopQA")

@dataclass
class ExperimentConfig:
    # --- Model ---
    model_name:  str         = "meta-llama/Llama-3.1-8B-Instruct"
    device_map:  str         = "auto"
    torch_dtype: torch.dtype = torch.bfloat16

    # --- Paths ---
    base_dir: str = _RESULTS_DIR

    # --- Dataset ---
    popqa_s_pop_threshold: int   = 500   # keep subjects with s_pop < this
    train_ratio:           float = 0.80  # 80% train, 10% val, 10% test
    seed:                  int   = 42

    # --- Extraction ---
    extraction_layer:  int  = 16
    extraction_pos:    str  = "resid_post"
    normalize_vectors: bool = True

    # --- Injection ---
    injection_layer: int = 16

    # --- Architecture ---
    architecture_id:    int = 4
    adapter_hidden_dim: int = 4096

    # --- Training ---
    use_lowercase: bool = True
    prompt_id:     int  = 1

    # --- Phase 1: Weighted CE + BERTTune ---
    phase1_epochs:        int   = 20
    phase1_lr:            float = 5e-4
    phase1_batch_size:    int   = 16
    phase1_dropout:       float = 0.15
    entity_token_weight:  float = 8.0
    berttune_lambda:      float = 3.0
    berttune_temperature: float = 1.0

    # --- Phase 2: GRPO ---
    grpo_epochs:           int   = 5
    grpo_lr:               float = 1e-5
    grpo_batch_size:       int   = 4
    grpo_num_generations:  int   = 4
    grpo_max_new_tokens:   int   = 64
    grpo_temperature:      float = 0.7
    grpo_kl_coeff:         float = 0.05
    grpo_max_examples:     int   = 600
    grpo_reward_weights: Dict[str, float] = field(default_factory=lambda: {
        "contains_entity":   0.5,
        "matches_reference": 0.5,
    })
    judge_model_name: str = "Qwen/Qwen2.5-3B-Instruct"

    # --- Multi-prompt concept extraction ---
    extraction_prompts: List[str] = field(default_factory=lambda: [
        "Q: What is {word}? A:",
        "Q: Describe {word}. A:",
        "Q: Tell me about {word}. A:",
        "Q: What do you know about {word}? A:",
    ])

    words: List[str] = field(default_factory=list)

    def get_safe_model_name(self):
        return self.model_name.replace("/", "_")

    @property
    def cache_dir(self):
        p = os.path.join(self.base_dir, "learning_cache")
        os.makedirs(p, exist_ok=True)
        return p

    @property
    def adapter_save_path(self):
        p = os.path.join(self.base_dir, "trained_adapters")
        os.makedirs(p, exist_ok=True)
        return os.path.join(p,
            f"adapter_{self.get_safe_model_name()}_L{self.injection_layer}_arch{self.architecture_id}.pt")

    @property
    def adapter_save_path_grpo(self):
        p = os.path.join(self.base_dir, "trained_adapters")
        os.makedirs(p, exist_ok=True)
        return os.path.join(p,
            f"adapter_{self.get_safe_model_name()}_L{self.injection_layer}_arch{self.architecture_id}_grpo.pt")

    @property
    def train_csv(self):
        return os.path.join(self.base_dir, "dataset", "train.csv")

    @property
    def val_csv(self):
        return os.path.join(self.base_dir, "dataset", "val.csv")

    @property
    def test_csv(self):
        return os.path.join(self.base_dir, "dataset", "test.csv")


cfg = ExperimentConfig()
os.makedirs(cfg.base_dir, exist_ok=True)
os.makedirs(os.path.join(cfg.base_dir, "dataset"), exist_ok=True)
print(f"Results directory : {cfg.base_dir}")
print(f"s_pop threshold   : {cfg.popqa_s_pop_threshold}")
print(f"Split             : 80% train / 10% val / 10% test")

Results directory : /storage/carmel/latent-rag/injecting-words/PopQA
s_pop threshold   : 500
Split             : 80% train / 10% val / 10% test


## Cell 3: Load & Filter PopQA

Load the full PopQA dataset from HuggingFace and filter to low-popularity subjects (`s_pop < threshold`).
The filtered candidates are saved to disk so this step only runs once.

In [3]:
# ── Load & Filter PopQA ───────────────────────────────────────────────────────
DATASET_DIR    = os.path.join(cfg.base_dir, "dataset")
CANDIDATES_CSV = os.path.join(DATASET_DIR, "filtered_candidates.csv")

if os.path.exists(CANDIDATES_CSV):
    print(f"Loading cached candidates: {CANDIDATES_CSV}")
    df_candidates = pd.read_csv(CANDIDATES_CSV)
    # possible_answers was stored as a Python list repr — parse it back
    df_candidates["answers_list"] = df_candidates["answers_list"].apply(
        lambda x: ast.literal_eval(x) if isinstance(x, str) else x
    )
else:
    print("Loading PopQA from HuggingFace (akariasai/PopQA)...")
    raw_ds  = load_dataset("akariasai/PopQA", split="test")  # PopQA ships as a single 'test' split
    df_raw  = raw_ds.to_pandas()
    print(f"Full PopQA: {len(df_raw):,} rows | columns: {list(df_raw.columns)}")

    # Filter by subject popularity
    df_pop = df_raw[df_raw["s_pop"] < cfg.popqa_s_pop_threshold].copy()
    print(f"After s_pop < {cfg.popqa_s_pop_threshold}: {len(df_pop):,} examples")

    # Parse possible_answers (may be stored as a JSON string or a Python list)
    def _parse_answers(val):
        if isinstance(val, list):
            return val
        try:
            return json.loads(val)
        except Exception:
            try:
                return ast.literal_eval(val)
            except Exception:
                return [str(val)]

    df_pop["answers_list"] = df_pop["possible_answers"].apply(_parse_answers)

    # Keep only the columns we need
    df_candidates = df_pop[["subj", "prop", "obj", "question", "answers_list", "s_pop", "o_pop"]].copy()
    df_candidates = df_candidates.reset_index(drop=True)
    df_candidates.to_csv(CANDIDATES_CSV, index=False)
    print(f"Saved {len(df_candidates):,} candidates → {CANDIDATES_CSV}")

print(f"\nCandidates: {len(df_candidates):,}")
print(f"Unique prop (relation) types:\n{df_candidates['prop'].value_counts().to_string()}")
df_candidates.head(3)

Loading PopQA from HuggingFace (akariasai/PopQA)...


Repo card metadata block was not found. Setting CardData to empty.


Full PopQA: 14,267 rows | columns: ['id', 'subj', 'prop', 'obj', 'subj_id', 'prop_id', 'obj_id', 's_aliases', 'o_aliases', 's_uri', 'o_uri', 's_wiki_title', 'o_wiki_title', 's_pop', 'o_pop', 'question', 'possible_answers']
After s_pop < 500: 5,451 examples
Saved 5,451 candidates → /storage/carmel/latent-rag/injecting-words/PopQA/dataset/filtered_candidates.csv

Candidates: 5,451
Unique prop (relation) types:
prop
author            845
director          715
genre             617
country           576
screenwriter      486
place of birth    447
sport             404
producer          336
occupation        313
composer          261
father            179
religion          121
capital            98
capital of         32
mother             21


,subj,prop,obj,question,answers_list,s_pop,o_pop
0,George Rankin,occupation,politician,What is George Rankin's occupation?,"[politician, political leader, political figur...",142,25692
1,John Mayne,occupation,journalist,What is John Mayne's occupation?,"[journalist, journo, journalists]",236,24952
2,Henry Feilden,occupation,politician,What is Henry Feilden's occupation?,"[politician, political leader, political figur...",58,25692


## Cell 4: Load Model & Tokenizer

Loads Llama-3.1-8B-Instruct (frozen) once. Used for both the no-context filtering step and the injector training.

In [4]:
# ── Load Model & Tokenizer ────────────────────────────────────────────────────
print("Loading tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(cfg.model_name)
tokenizer.padding_side = "right"
tokenizer.pad_token    = tokenizer.eos_token

print("Loading model...")
model = AutoModelForCausalLM.from_pretrained(
    cfg.model_name,
    device_map=cfg.device_map,
    dtype=cfg.torch_dtype,
)
model.eval()
for p in model.parameters():
    p.requires_grad = False

print(f"Model loaded. Hidden size: {model.config.hidden_size}")
print(f"Layers: {model.config.num_hidden_layers}")
print(f"Device: {model.device}")

Loading tokenizer...
Loading model...


/opt/conda/lib/python3.10/site-packages/torchvision/io/image.py:13: UserWarning: Failed to load image Python extension: 'Could not load this library: /opt/conda/lib/python3.10/site-packages/torchvision/image.so'If you don't plan on using image functionality from `torchvision.io`, you can ignore this warning. Otherwise, there might be something wrong with your environment. Did you have `libjpeg` or `libpng` installed before building `torchvision` from source?
  warn(
Loading weights: 100%|██████████| 291/291 [00:01<00:00, 161.26it/s]


Model loaded. Hidden size: 4096
Layers: 32
Device: cuda:0


## Cell 5: No-Context Filtering

Run the base model (no adapter, no context) on every candidate.
Keep only examples where the model's output does **not** contain any of the `possible_answers`.
This ensures all retained examples are genuinely unknown to the base model.

Results are cached to `no_context_check.csv` so this only runs once.

In [5]:
# ── No-Context Filtering ──────────────────────────────────────────────────────
NO_CTX_CSV = os.path.join(DATASET_DIR, "no_context_check.csv")


def _model_knows(generated: str, answers_list: list) -> bool:
    """Return True if the generated text contains any accepted answer (case-insensitive)."""
    gen_lower = generated.lower()
    return any(str(ans).lower() in gen_lower for ans in answers_list)


@torch.inference_mode()
def _generate_no_ctx(question: str, max_new_tokens: int = 64) -> str:
    msgs = [{"role": "user", "content": question}]
    prompt_str = tokenizer.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(prompt_str, return_tensors="pt").to(model.device)
    out_ids = model.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        do_sample=False,
        pad_token_id=tokenizer.eos_token_id,
    )
    return tokenizer.decode(out_ids[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)


if os.path.exists(NO_CTX_CSV):
    print(f"Loading cached no-context results: {NO_CTX_CSV}")
    df_no_ctx = pd.read_csv(NO_CTX_CSV)
    df_no_ctx["answers_list"] = df_no_ctx["answers_list"].apply(
        lambda x: ast.literal_eval(x) if isinstance(x, str) else x
    )
else:
    print(f"Running no-context evaluation on {len(df_candidates):,} candidates...")
    records = []
    for _, row in tqdm(df_candidates.iterrows(), total=len(df_candidates), desc="No-context"):
        generated = _generate_no_ctx(row["question"])
        records.append({
            "subj":        row["subj"],
            "prop":        row["prop"],
            "obj":         row["obj"],
            "question":    row["question"],
            "answers_list": row["answers_list"],
            "s_pop":       row["s_pop"],
            "generated":   generated,
            "model_knew":  _model_knows(generated, row["answers_list"]),
        })
    df_no_ctx = pd.DataFrame(records)
    df_no_ctx.to_csv(NO_CTX_CSV, index=False)
    print(f"Saved no-context results: {NO_CTX_CSV}")

total     = len(df_no_ctx)
knew      = df_no_ctx["model_knew"].sum()
did_not   = total - knew
print(f"\nTotal candidates : {total:,}")
print(f"Model knew       : {knew:,}  ({100*knew/total:.1f}%)")
print(f"Model did NOT know: {did_not:,}  ({100*did_not/total:.1f}%) — these form our dataset")

Running no-context evaluation on 5,451 candidates...


No-context: 100%|██████████| 5451/5451 [42:29<00:00,  2.14it/s]  

Saved no-context results: /storage/carmel/latent-rag/injecting-words/PopQA/dataset/no_context_check.csv

Total candidates : 5,451
Model knew       : 547  (10.0%)
Model did NOT know: 4,904  (90.0%) — these form our dataset


## Cell 6: Save Train / Test Splits

Filter to examples the model could not answer, then stratify-split by `prop` (80/20).
Saved CSVs are the canonical dataset used in all downstream cells.

In [6]:
# ── Save Train / Val / Test Splits (80 / 10 / 10) ────────────────────────────
TRAIN_CSV = cfg.train_csv
VAL_CSV   = cfg.val_csv
TEST_CSV  = cfg.test_csv

if os.path.exists(TRAIN_CSV) and os.path.exists(VAL_CSV) and os.path.exists(TEST_CSV):
    print("Splits already exist — skipping.")
    print(f"  Train : {TRAIN_CSV}")
    print(f"  Val   : {VAL_CSV}")
    print(f"  Test  : {TEST_CSV}")
else:
    df_unknown = df_no_ctx[~df_no_ctx["model_knew"]].copy().reset_index(drop=True)
    print(f"Filtered to {len(df_unknown):,} unknown examples.")

    # Simple random shuffle then slice: 80 / 10 / 10
    df_unknown = df_unknown.sample(frac=1, random_state=cfg.seed).reset_index(drop=True)
    n       = len(df_unknown)
    n_train = int(round(n * 0.80))
    n_val   = int(round(n * 0.10))
    # test gets the remainder so no rows are wasted
    df_train = df_unknown.iloc[:n_train]
    df_val   = df_unknown.iloc[n_train : n_train + n_val]
    df_test  = df_unknown.iloc[n_train + n_val:]

    keep_cols = ["subj", "prop", "obj", "question", "answers_list"]
    df_train[keep_cols].to_csv(TRAIN_CSV, index=False)
    df_val[keep_cols].to_csv(VAL_CSV,     index=False)
    df_test[keep_cols].to_csv(TEST_CSV,   index=False)
    print(f"Saved train ({len(df_train):,}) → {TRAIN_CSV}")
    print(f"Saved val   ({len(df_val):,})   → {VAL_CSV}")
    print(f"Saved test  ({len(df_test):,})  → {TEST_CSV}")

print("\nProp distribution — train:")
print(pd.read_csv(TRAIN_CSV)["prop"].value_counts().to_string())

Filtered to 4,904 unknown examples.
Saved train (3,923) → /storage/carmel/latent-rag/injecting-words/PopQA/dataset/train.csv
Saved val   (490)   → /storage/carmel/latent-rag/injecting-words/PopQA/dataset/val.csv
Saved test  (491)  → /storage/carmel/latent-rag/injecting-words/PopQA/dataset/test.csv

Prop distribution — train:
prop
author            638
director          563
genre             460
screenwriter      369
place of birth    369
sport             278
producer          258
occupation        251
country           233
composer          189
father            145
religion           99
capital            36
capital of         21
mother             14


## Cell 7: Load Splits (Restart Checkpoint)

If the GPU session was restarted after the dataset was prepared, start here.
Re-runs Cells 1–2 (imports + config) and 4 (model load) first, then run this cell.

In [7]:
# ── Load Train / Val / Test Splits ────────────────────────────────────────────
# Restart checkpoint: re-run Cells 1–2 (imports + config) and Cell 4 (model),
# then run this cell to restore all data splits from disk.

def _load_split(csv_path: str) -> List[dict]:
    df = pd.read_csv(csv_path)
    df["answers_list"] = df["answers_list"].apply(
        lambda x: ast.literal_eval(x) if isinstance(x, str) else x
    )
    records = []
    for _, row in df.iterrows():
        obj  = str(row["obj"])
        prop = str(row["prop"])
        records.append({
            "entity_key":      obj.lower() if cfg.use_lowercase else obj,
            "entity_original": obj,
            "category_key":    prop.lower() if cfg.use_lowercase else prop,
            "question":        str(row["question"]),
            "answer":          str(row["answers_list"][0]) if row["answers_list"] else obj,
            "answers_list":    row["answers_list"],
        })
    return records


train_data = _load_split(cfg.train_csv)
val_data   = _load_split(cfg.val_csv)
test_data  = _load_split(cfg.test_csv)

# Unique obj values in train (for concept vector extraction)
cfg.words = list({item["entity_original"] for item in train_data})

print(f"Train : {len(train_data):,} examples  |  unique obj: {len(cfg.words):,}")
print(f"Val   : {len(val_data):,} examples")
print(f"Test  : {len(test_data):,} examples")

Train : 3,923 examples  |  unique obj: 2,385
Val   : 490 examples
Test  : 491 examples


## Cell 8: Concept Vector Extraction

For each unique `obj` value (the answer entity) in the training set, extract the model's hidden-state representation using 4 QA-style prompts and average them.
Vectors are cached to disk.

In [8]:
# ── Concept Vector Extraction ─────────────────────────────────────────────────

@torch.inference_mode()
def get_layer_activation(model, tokenizer, prompt: str,
                         layer_idx: int, which: str = "resid_post") -> torch.Tensor:
    enc = tokenizer(prompt, return_tensors="pt", truncation=True, add_special_tokens=True)
    enc = {k: v.to(model.device) for k, v in enc.items()}
    out = model(**enc, output_hidden_states=True, use_cache=False)
    hs  = out.hidden_states
    acts = hs[layer_idx + 1] if which == "resid_post" else hs[layer_idx]
    length = enc["attention_mask"].sum(dim=1) - 1
    return acts[0, length[0]].detach().cpu()


def get_concept_vectors(cfg: ExperimentConfig, model, tokenizer,
                        words: List[str], split_tag: str = "train") -> dict:
    norm_tag = "norm" if cfg.normalize_vectors else "raw"
    filename = (f"{split_tag}_concepts_{cfg.get_safe_model_name()}"
                f"_L{cfg.extraction_layer}_{cfg.extraction_pos}_{norm_tag}_multiQA.pt")
    filepath = os.path.join(cfg.cache_dir, filename)

    if os.path.exists(filepath):
        print(f"Loading concept vectors from cache: {filepath}")
        return torch.load(filepath, map_location="cpu")["vectors"]

    print(f"Computing concept vectors for {len(set(words))} unique entities...")
    concept_vecs = {}
    for word in tqdm(set(words), desc="Concept vectors"):
        key  = word.lower() if cfg.use_lowercase else word
        vecs = []
        for tmpl in cfg.extraction_prompts:
            prompt = tmpl.replace("{word}", word)
            vec = get_layer_activation(model, tokenizer, prompt,
                                       cfg.extraction_layer, cfg.extraction_pos)
            vecs.append(vec)
        avg_vec = torch.stack(vecs).mean(dim=0)
        if cfg.normalize_vectors:
            avg_vec = F.normalize(avg_vec, dim=-1)
        concept_vecs[key] = avg_vec

    torch.save({"words": list(concept_vecs.keys()), "vectors": concept_vecs}, filepath)
    print(f"Saved concept vectors: {filepath}")
    return concept_vecs


# Extract for train (unique obj values) and test (unique obj values)
test_words = list({item["entity_original"] for item in test_data})

concept_vecs      = get_concept_vectors(cfg, model, tokenizer, cfg.words,  split_tag="popqa_train")
concept_vecs_test = get_concept_vectors(cfg, model, tokenizer, test_words, split_tag="popqa_test")
concept_vecs_all  = {**concept_vecs, **concept_vecs_test}

print(f"Train concept vectors: {len(concept_vecs)}")
print(f"Test  concept vectors: {len(concept_vecs_test)}")

Computing concept vectors for 2385 unique entities...


Concept vectors: 100%|██████████| 2385/2385 [02:37<00:00, 15.11it/s]


Saved concept vectors: /storage/carmel/latent-rag/injecting-words/PopQA/learning_cache/popqa_train_concepts_meta-llama_Llama-3.1-8B-Instruct_L16_resid_post_norm_multiQA.pt
Computing concept vectors for 389 unique entities...


Concept vectors: 100%|██████████| 389/389 [00:25<00:00, 15.17it/s]

Saved concept vectors: /storage/carmel/latent-rag/injecting-words/PopQA/learning_cache/popqa_test_concepts_meta-llama_Llama-3.1-8B-Instruct_L16_resid_post_norm_multiQA.pt
Train concept vectors: 2385
Test  concept vectors: 389


## Cell 9: Adapter Architecture (V4)

2-layer MLP: `input_dim → hidden_dim → GELU → Dropout → input_dim`. Same as V4.

In [9]:
# ── Adapter Architecture (V4) ─────────────────────────────────────────────────

class InjectionAdapterV4(nn.Module):
    def __init__(self, input_dim: int, hidden_dim: int = 4096, dropout: float = 0.1):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, input_dim),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.net(x)


def build_adapter(cfg: ExperimentConfig, model) -> nn.Module:
    input_dim = model.config.hidden_size
    adapter   = InjectionAdapterV4(
        input_dim  = input_dim,
        hidden_dim = cfg.adapter_hidden_dim,
        dropout    = cfg.phase1_dropout,
    )
    n_params = sum(p.numel() for p in adapter.parameters())
    print(f"Adapter (arch {cfg.architecture_id}): {n_params:,} parameters")
    return adapter

## Cell 10: Training Helpers

BERTTune semantic loss, entity span detection, injection hook, and batch builder — identical to V4.
The only difference: `entity_original` = `obj` (a short answer string, often 1–3 tokens).

In [10]:
# ── BERTTune Semantic Loss ────────────────────────────────────────────────────

def berttune_semantic_loss(
    logits:       torch.Tensor,
    target_ids:   torch.Tensor,
    embed_matrix: torch.Tensor,
    answer_mask:  torch.Tensor,
    temperature:  float = 1.0,
) -> torch.Tensor:
    B, T, V = logits.shape
    probs     = F.softmax(logits.float() / temperature, dim=-1)
    soft_embs = probs @ embed_matrix.float()
    with torch.no_grad():
        ref_embs = embed_matrix.float()[target_ids.clamp(min=0)]
    mask      = answer_mask.float().unsqueeze(-1)
    soft_embs = soft_embs * mask
    ref_embs  = ref_embs  * mask
    denom     = mask.sum(dim=1).clamp(min=1e-8)
    soft_mean = soft_embs.sum(dim=1) / denom
    ref_mean  = ref_embs.sum(dim=1)  / denom
    cos_sim   = F.cosine_similarity(soft_mean, ref_mean, dim=-1)
    return 1.0 - cos_sim.mean()


# ── Training Helpers ──────────────────────────────────────────────────────────

def find_entity_token_spans(answer_ids: List[int], entity_ids: List[int]) -> List[int]:
    positions = []
    n, m = len(answer_ids), len(entity_ids)
    if m == 0:
        return positions
    for i in range(n - m + 1):
        if answer_ids[i : i + m] == entity_ids:
            positions.extend(range(i, i + m))
    return positions


def make_injection_hook(injection_vecs: torch.Tensor, injection_indices: List[int]):
    def hook(module, args, output):
        hs = output[0] if isinstance(output, tuple) else output
        for b, idx in enumerate(injection_indices):
            if b < hs.shape[0] and idx < hs.shape[1]:
                hs[b, idx, :] = hs[b, idx, :] + injection_vecs[b]
        return (hs,) + output[1:] if isinstance(output, tuple) else hs
    return hook


def build_batch(items, tokenizer, cfg, concept_vecs):
    input_ids_list, labels_list, weights_list = [], [], []
    answer_masks_list, injection_indices, batch_vectors = [], [], []

    for item in items:
        vec = concept_vecs.get(item["entity_key"])
        if vec is None:
            continue

        msgs       = [{"role": "user", "content": item["question"]}]
        prompt_str = tokenizer.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
        answer_str = item["answer"] + tokenizer.eos_token

        prompt_ids = tokenizer.encode(prompt_str, add_special_tokens=False)
        ans_ids    = tokenizer.encode(answer_str, add_special_tokens=False)

        entity_ids = tokenizer.encode(item["entity_original"], add_special_tokens=False)
        entity_pos = set(find_entity_token_spans(ans_ids, entity_ids))
        weights    = [cfg.entity_token_weight if i in entity_pos else 1.0
                      for i in range(len(ans_ids))]

        full_ids = prompt_ids + ans_ids
        lbl_ids  = [-100] * len(prompt_ids) + ans_ids
        lbl_wts  = [0.0]  * len(prompt_ids) + weights
        ans_mask = [0.0]  * len(prompt_ids) + [1.0] * len(ans_ids)

        input_ids_list.append(torch.tensor(full_ids,   dtype=torch.long))
        labels_list.append(torch.tensor(lbl_ids,       dtype=torch.long))
        weights_list.append(torch.tensor(lbl_wts,      dtype=torch.float))
        answer_masks_list.append(torch.tensor(ans_mask, dtype=torch.float))
        injection_indices.append(len(prompt_ids) - 1)
        batch_vectors.append(vec)

    if not input_ids_list:
        return None

    pad = tokenizer.pad_token_id
    input_ids    = torch.nn.utils.rnn.pad_sequence(input_ids_list,    batch_first=True, padding_value=pad)
    labels       = torch.nn.utils.rnn.pad_sequence(labels_list,       batch_first=True, padding_value=-100)
    weights      = torch.nn.utils.rnn.pad_sequence(weights_list,      batch_first=True, padding_value=0.0)
    answer_masks = torch.nn.utils.rnn.pad_sequence(answer_masks_list, batch_first=True, padding_value=0.0)
    attn_mask    = (input_ids != pad).long()

    return {
        "input_ids":        input_ids,
        "labels":           labels,
        "weights":          weights,
        "answer_masks":     answer_masks,
        "attention_mask":   attn_mask,
        "injection_indices": injection_indices,
        "batch_vectors":    torch.stack(batch_vectors),
    }

## Cell 11: Phase 1 Training (Weighted CE + BERTTune)

Same as V4. Early stopping on validation loss with patience=3.

In [11]:
# ── Phase 1: Weighted CE + BERTTune Training ──────────────────────────────────

def train_phase1(cfg, model, tokenizer, adapter, train_data, val_data, concept_vecs):
    optimizer    = torch.optim.AdamW(adapter.parameters(), lr=cfg.phase1_lr)
    embed_matrix = model.model.embed_tokens.weight.detach()
    layer        = model.model.layers[cfg.injection_layer]
    history      = {"train_ce": [], "train_sem": [], "train_total": [], "val_loss": []}

    n_batches_total = (len(train_data) + cfg.phase1_batch_size - 1) // cfg.phase1_batch_size
    best_val, patience, wait, best_state = float("inf"), 3, 0, None

    for epoch in range(cfg.phase1_epochs):
        adapter.train()
        random.shuffle(train_data)
        total_ce, total_sem, total_loss, n_batches = 0., 0., 0., 0

        pbar = tqdm(range(0, len(train_data), cfg.phase1_batch_size),
                    desc=f"Epoch {epoch+1:02d}/{cfg.phase1_epochs}",
                    total=n_batches_total)

        for i in pbar:
            batch = build_batch(train_data[i : i + cfg.phase1_batch_size],
                                tokenizer, cfg, concept_vecs)
            if batch is None:
                continue

            dev          = model.device
            input_ids    = batch["input_ids"].to(dev)
            labels       = batch["labels"].to(dev)
            weights      = batch["weights"].to(dev)
            answer_masks = batch["answer_masks"].to(dev)
            attn_mask    = batch["attention_mask"].to(dev)

            raw_vecs       = batch["batch_vectors"].to(dev, dtype=cfg.torch_dtype)
            injection_vecs = adapter(raw_vecs)

            hook   = make_injection_hook(injection_vecs, batch["injection_indices"])
            handle = layer.register_forward_hook(hook)
            try:
                outputs = model(input_ids=input_ids, attention_mask=attn_mask, use_cache=False)
            finally:
                handle.remove()

            logits = outputs.logits
            shift_logits  = logits[:, :-1, :].contiguous()
            shift_labels  = labels[:, 1:].contiguous()
            shift_weights = weights[:, 1:].contiguous()
            shift_amask   = answer_masks[:, 1:].contiguous()
            B, T, V = shift_logits.shape

            per_token_ce = F.cross_entropy(
                shift_logits.view(B * T, V),
                shift_labels.view(B * T),
                ignore_index=-100,
                reduction="none",
            ).view(B, T)
            loss_ce = (per_token_ce * shift_weights).sum() / (shift_weights.sum() + 1e-8)

            valid_answer_mask = shift_amask * (shift_labels != -100).float()
            loss_sem = berttune_semantic_loss(
                logits       = shift_logits,
                target_ids   = shift_labels.clamp(min=0),
                embed_matrix = embed_matrix,
                answer_mask  = valid_answer_mask,
                temperature  = cfg.berttune_temperature,
            )

            loss = loss_ce + cfg.berttune_lambda * loss_sem
            optimizer.zero_grad()
            loss.backward()
            torch.nn.utils.clip_grad_norm_(adapter.parameters(), 1.0)
            optimizer.step()

            total_ce   += loss_ce.item()
            total_sem  += loss_sem.item()
            total_loss += loss.item()
            n_batches  += 1
            pbar.set_postfix(ce=f"{loss_ce.item():.4f}", sem=f"{loss_sem.item():.4f}")

        # Validation
        adapter.eval()
        val_losses = []
        for i in range(0, len(val_data), cfg.phase1_batch_size):
            batch = build_batch(val_data[i : i + cfg.phase1_batch_size],
                                tokenizer, cfg, concept_vecs)
            if batch is None:
                continue
            dev       = model.device
            input_ids = batch["input_ids"].to(dev)
            labels    = batch["labels"].to(dev)
            weights   = batch["weights"].to(dev)
            attn_mask = batch["attention_mask"].to(dev)
            raw_vecs  = batch["batch_vectors"].to(dev, dtype=cfg.torch_dtype)
            with torch.no_grad():
                injection_vecs = adapter(raw_vecs)
                hook   = make_injection_hook(injection_vecs, batch["injection_indices"])
                handle = layer.register_forward_hook(hook)
                try:
                    outputs = model(input_ids=input_ids, attention_mask=attn_mask, use_cache=False)
                finally:
                    handle.remove()
            logits = outputs.logits
            shift_logits = logits[:, :-1, :].contiguous()
            shift_labels = labels[:, 1:].contiguous()
            shift_weights = weights[:, 1:].contiguous()
            B2, T2, V2 = shift_logits.shape
            per_token_ce = F.cross_entropy(
                shift_logits.view(B2 * T2, V2),
                shift_labels.view(B2 * T2),
                ignore_index=-100, reduction="none",
            ).view(B2, T2)
            val_losses.append((per_token_ce * shift_weights).sum().item() /
                              (shift_weights.sum().item() + 1e-8))

        val_loss = np.mean(val_losses) if val_losses else float("inf")
        avg_ce  = total_ce  / max(n_batches, 1)
        avg_sem = total_sem / max(n_batches, 1)
        avg_tot = total_loss / max(n_batches, 1)
        history["train_ce"].append(avg_ce)
        history["train_sem"].append(avg_sem)
        history["train_total"].append(avg_tot)
        history["val_loss"].append(val_loss)

        print(f"  Epoch {epoch+1:02d}: train={avg_tot:.4f} (CE={avg_ce:.4f}, sem={avg_sem:.4f})  val={val_loss:.4f}")

        if val_loss < best_val:
            best_val   = val_loss
            best_state = copy.deepcopy(adapter.state_dict())
            wait       = 0
        else:
            wait += 1
            if wait >= patience:
                print(f"  Early stopping at epoch {epoch+1}.")
                break

    if best_state is not None:
        adapter.load_state_dict(best_state)
        print(f"Restored best adapter (val_loss={best_val:.4f})")
    return history

## Cell 12: Qwen Judge for GRPO Reward

Same as V4. Local Qwen2.5-3B-Instruct used as the reward model during GRPO training.

In [12]:
# ── Qwen Judge for GRPO Reward ────────────────────────────────────────────────

def load_qwen_judge(cfg: ExperimentConfig):
    print(f"Loading Qwen judge: {cfg.judge_model_name}")
    judge_tok = AutoTokenizer.from_pretrained(cfg.judge_model_name)
    judge_tok.pad_token = judge_tok.eos_token
    judge_mdl = AutoModelForCausalLM.from_pretrained(
        cfg.judge_model_name, dtype=torch.bfloat16, device_map="auto"
    )
    judge_mdl.eval()
    for param in judge_mdl.parameters():
        param.requires_grad = False
    print("Qwen judge loaded.")
    return judge_mdl, judge_tok


def qwen_judge_reward(question: str, entity: str, gold_answer: str,
                      generated_answer: str, cfg: ExperimentConfig,
                      judge_model, judge_tokenizer) -> float:
    prompt = (
        "You are an expert evaluator for a question-answering system.\n"
        "Analyze the following and return a JSON object.\n\n"
        "**Input:**\n"
        f"- Entity: \"{entity}\"\n"
        f"- Question: \"{question}\"\n"
        f"- Gold Answer: \"{gold_answer}\"\n"
        f"- Generated Answer: \"{generated_answer}\"\n\n"
        "**Task:** Evaluate on these 2 metrics (0.0=no, 1.0=yes):\n"
        f"1. contains_entity: Does the generated answer mention \"{entity}\"?\n"
        "2. matches_reference: Does the generated answer convey the same core info as the gold answer?\n\n"
        "**Output:** Return ONLY a JSON object: {\"contains_entity\": float, \"matches_reference\": float}"
    )
    messages = [
        {"role": "system", "content": "You are a strict JSON output machine. Output only valid JSON."},
        {"role": "user",   "content": prompt},
    ]
    text   = judge_tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = judge_tokenizer(text, return_tensors="pt").to(judge_model.device)
    with torch.no_grad():
        outputs = judge_model.generate(
            **inputs, max_new_tokens=80, do_sample=False,
            pad_token_id=judge_tokenizer.eos_token_id,
        )
    response = judge_tokenizer.decode(
        outputs[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True
    )
    try:
        raw = response.strip().lstrip("```json").lstrip("```").rstrip("```").strip()
        result = json.loads(raw)
        ce = float(result.get("contains_entity",  0.0))
        mr = float(result.get("matches_reference", 0.0))
        return (cfg.grpo_reward_weights["contains_entity"]  * ce +
                cfg.grpo_reward_weights["matches_reference"] * mr)
    except Exception:
        return 0.0

## Cell 13: Phase 2 — GRPO Training

Group Relative Policy Optimization: generate G completions per prompt, score with Qwen judge,
compute group-relative advantages, apply REINFORCE policy gradient + KL penalty vs Phase 1 checkpoint.

In [13]:
# ── Phase 2: GRPO ─────────────────────────────────────────────────────────────

def generate_with_injection(model, tokenizer, adapter, item, concept_vecs, cfg,
                            num_return: int = 1, temperature: float = 0.7):
    vec = concept_vecs.get(item["entity_key"])
    if vec is None:
        return [""] * num_return

    layer      = model.model.layers[cfg.injection_layer]
    msgs       = [{"role": "user", "content": item["question"]}]
    prompt_str = tokenizer.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
    inputs     = tokenizer(prompt_str, return_tensors="pt").to(model.device)

    with torch.no_grad():
        injection_vec = adapter(
            vec.unsqueeze(0).to(model.device, dtype=cfg.torch_dtype)
        ).squeeze(0)

    inject_idx = inputs["input_ids"].shape[1] - 1

    def _hook(module, args, output):
        hs = output[0] if isinstance(output, tuple) else output
        if inject_idx < hs.shape[1]:
            hs[0, inject_idx, :] = hs[0, inject_idx, :] + injection_vec
        return (hs,) + output[1:] if isinstance(output, tuple) else hs

    generations = []
    for _ in range(num_return):
        handle = layer.register_forward_hook(_hook)
        try:
            with torch.no_grad():
                out_ids = model.generate(
                    **inputs,
                    max_new_tokens=cfg.grpo_max_new_tokens,
                    do_sample=(temperature > 0),
                    temperature=temperature if temperature > 0 else None,
                    pad_token_id=tokenizer.eos_token_id,
                )
        finally:
            handle.remove()
        gen_text = tokenizer.decode(
            out_ids[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True
        )
        generations.append(gen_text)
    return generations


def compute_log_probs(model, tokenizer, adapter, item, generated_text, concept_vecs, cfg):
    vec = concept_vecs.get(item["entity_key"])
    if vec is None:
        return torch.tensor(0.0, device=model.device)

    layer      = model.model.layers[cfg.injection_layer]
    msgs       = [{"role": "user", "content": item["question"]}]
    prompt_str = tokenizer.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
    answer_str = generated_text + tokenizer.eos_token

    prompt_ids = tokenizer.encode(prompt_str, add_special_tokens=False)
    ans_ids    = tokenizer.encode(answer_str, add_special_tokens=False)
    full_ids   = torch.tensor(prompt_ids + ans_ids, dtype=torch.long).unsqueeze(0).to(model.device)

    injection_vec = adapter(
        vec.unsqueeze(0).to(model.device, dtype=cfg.torch_dtype)
    ).squeeze(0)
    inject_idx = len(prompt_ids) - 1

    def _hook(module, args, output):
        hs = output[0] if isinstance(output, tuple) else output
        if inject_idx < hs.shape[1]:
            hs[0, inject_idx, :] = hs[0, inject_idx, :] + injection_vec
        return (hs,) + output[1:] if isinstance(output, tuple) else hs

    handle = layer.register_forward_hook(_hook)
    try:
        outputs = model(input_ids=full_ids, use_cache=False)
    finally:
        handle.remove()

    logits  = outputs.logits[:, :-1, :]          # (1, T-1, V)
    targets = full_ids[:, 1:]                    # (1, T-1)
    log_probs   = F.log_softmax(logits.float(), dim=-1)
    token_lps   = log_probs.gather(2, targets.unsqueeze(-1)).squeeze(-1)  # (1, T-1)
    answer_start = len(prompt_ids) - 1
    return token_lps[0, answer_start:].sum()


def build_grpo_data(train_data: List[dict], cfg: ExperimentConfig) -> List[dict]:
    """Sample up to grpo_max_examples from train_data for GRPO."""
    data = list(train_data)
    random.seed(cfg.seed)
    random.shuffle(data)
    return data[:cfg.grpo_max_examples]


def train_grpo(cfg, model, tokenizer, adapter, grpo_data, concept_vecs,
               judge_model, judge_tokenizer):
    optimizer   = torch.optim.AdamW(adapter.parameters(), lr=cfg.grpo_lr)
    layer       = model.model.layers[cfg.injection_layer]
    history     = {"reward_mean": [], "reward_std": [], "policy_loss": [], "kl_loss": []}
    ref_adapter = copy.deepcopy(adapter)
    ref_adapter.eval()

    for epoch in range(cfg.grpo_epochs):
        adapter.train()
        random.shuffle(grpo_data)
        epoch_rewards, epoch_pol, epoch_kl = [], [], []

        pbar = tqdm(range(0, len(grpo_data), cfg.grpo_batch_size),
                    desc=f"GRPO Epoch {epoch+1}/{cfg.grpo_epochs}")
        for i in pbar:
            batch_items = grpo_data[i : i + cfg.grpo_batch_size]
            total_pol = torch.tensor(0.0, device=model.device, requires_grad=True)
            total_kl  = torch.tensor(0.0, device=model.device)
            n_valid   = 0

            for item in batch_items:
                # Generate G completions
                gens = generate_with_injection(
                    model, tokenizer, adapter, item, concept_vecs, cfg,
                    num_return=cfg.grpo_num_generations,
                    temperature=cfg.grpo_temperature,
                )
                # Score
                rewards = [
                    qwen_judge_reward(
                        item["question"], item["entity_original"],
                        item["answer"], g, cfg, judge_model, judge_tokenizer
                    )
                    for g in gens
                ]
                rewards_t = torch.tensor(rewards, dtype=torch.float)
                mean_r    = rewards_t.mean()
                std_r     = rewards_t.std().clamp(min=1e-8)
                advantages = (rewards_t - mean_r) / std_r
                epoch_rewards.extend(rewards)

                for gen, adv in zip(gens, advantages.tolist()):
                    if abs(adv) < 1e-6 or not gen.strip():
                        continue
                    lp_cur = compute_log_probs(model, tokenizer, adapter,
                                              item, gen, concept_vecs, cfg)
                    with torch.no_grad():
                        lp_ref = compute_log_probs(model, tokenizer, ref_adapter,
                                                   item, gen, concept_vecs, cfg)
                    pol_loss = -adv * lp_cur
                    kl       = (lp_cur - lp_ref).clamp(min=0).detach()
                    total_pol = total_pol + pol_loss
                    total_kl  = total_kl  + kl
                    n_valid  += 1

            if n_valid > 0:
                loss = (total_pol + cfg.grpo_kl_coeff * total_kl) / n_valid
                optimizer.zero_grad()
                loss.backward()
                torch.nn.utils.clip_grad_norm_(adapter.parameters(), 1.0)
                optimizer.step()
                epoch_pol.append(total_pol.item() / n_valid)
                epoch_kl.append(total_kl.item()   / n_valid)
                pbar.set_postfix(reward=f"{mean_r.item():.3f}")

        history["reward_mean"].append(float(np.mean(epoch_rewards)) if epoch_rewards else 0.0)
        history["reward_std"].append(float(np.std(epoch_rewards))   if epoch_rewards else 0.0)
        history["policy_loss"].append(float(np.mean(epoch_pol)) if epoch_pol else 0.0)
        history["kl_loss"].append(float(np.mean(epoch_kl))      if epoch_kl  else 0.0)
        print(f"  GRPO Epoch {epoch+1}: reward={history['reward_mean'][-1]:.3f} "
              f"± {history['reward_std'][-1]:.3f}")

    return history

## Cell 14: Run Training (Both Phases)

In [ ]:
# ── Run Training ──────────────────────────────────────────────────────────────
adapter = build_adapter(cfg, model).to(model.device, dtype=cfg.torch_dtype)

print(f"Phase 1: Weighted CE + BERTTune ({cfg.phase1_epochs} epochs)")
print(f"  Training examples : {len(train_data)}")
print(f"  Val examples      : {len(val_data)}")
print(f"  Entity weight     : {cfg.entity_token_weight}x")
print(f"  BERTTune lambda   : {cfg.berttune_lambda}")

phase1_history = train_phase1(cfg, model, tokenizer, adapter,
                              train_data, val_data, concept_vecs)
torch.save(adapter.state_dict(), cfg.adapter_save_path)
print(f"Phase 1 adapter saved: {cfg.adapter_save_path}")

# ── Phase 2: GRPO ─────────────────────────────────────────────────────────────
grpo_data = build_grpo_data(train_data, cfg)  # sample from full train split
print(f"\nPhase 2: GRPO ({cfg.grpo_epochs} epochs)")
print(f"  GRPO examples       : {len(grpo_data)}")
print(f"  Generations/prompt  : {cfg.grpo_num_generations}")
print(f"  Temperature         : {cfg.grpo_temperature}")
print(f"  KL coeff            : {cfg.grpo_kl_coeff}")

judge_model, judge_tokenizer = load_qwen_judge(cfg)

grpo_history = train_grpo(cfg, model, tokenizer, adapter, grpo_data,
                          concept_vecs_all, judge_model, judge_tokenizer)
torch.save(adapter.state_dict(), cfg.adapter_save_path_grpo)
print(f"GRPO adapter saved: {cfg.adapter_save_path_grpo}")

del judge_model, judge_tokenizer
torch.cuda.empty_cache()
print("Qwen judge freed from GPU memory.")

Adapter (arch 4): 33,562,624 parameters
Phase 1: Weighted CE + BERTTune (20 epochs)
  Training examples : 3923
  Val examples      : 490
  Entity weight     : 8.0x
  BERTTune lambda   : 3.0


Epoch 01/20:  61%|██████▏   | 151/246 [00:16<00:10,  9.45it/s, ce=2.0487, sem=0.2695]

## Cell 15: Training Loss Plots

In [ ]:
# ── Training Loss Plots ───────────────────────────────────────────────────────
PLOTS_DIR = os.path.join(cfg.base_dir, "plots")
os.makedirs(PLOTS_DIR, exist_ok=True)

fig, axes = plt.subplots(2, 2, figsize=(14, 10))

epochs_p1 = range(1, len(phase1_history["train_total"]) + 1)
axes[0][0].plot(epochs_p1, phase1_history["train_ce"],  label="Train CE",      color="steelblue")
axes[0][0].plot(epochs_p1, phase1_history["train_sem"], label="Train Semantic", color="darkorange")
axes[0][0].set_title("Phase 1: Loss Components"); axes[0][0].legend(); axes[0][0].grid(alpha=0.3)

axes[0][1].plot(epochs_p1, phase1_history["train_total"], label="Train", color="steelblue")
axes[0][1].plot(epochs_p1, phase1_history["val_loss"],    label="Val",   color="tomato")
axes[0][1].set_title("Phase 1: Train vs Val"); axes[0][1].legend(); axes[0][1].grid(alpha=0.3)

epochs_p2 = range(1, len(grpo_history["reward_mean"]) + 1)
r_mean, r_std = grpo_history["reward_mean"], grpo_history["reward_std"]
axes[1][0].plot(epochs_p2, r_mean, color="seagreen", marker="o", label="Mean Reward")
axes[1][0].fill_between(epochs_p2,
    [m - s for m, s in zip(r_mean, r_std)],
    [m + s for m, s in zip(r_mean, r_std)],
    alpha=0.2, color="seagreen")
axes[1][0].set_title("Phase 2 GRPO: Mean Reward"); axes[1][0].legend(); axes[1][0].grid(alpha=0.3)

axes[1][1].plot(epochs_p2, grpo_history["policy_loss"], label="Policy Loss", color="steelblue")
axes[1][1].plot(epochs_p2, grpo_history["kl_loss"],     label="KL Loss",     color="tomato")
axes[1][1].set_title("Phase 2 GRPO: Policy & KL Loss"); axes[1][1].legend(); axes[1][1].grid(alpha=0.3)

for ax_row in axes:
    for ax in ax_row:
        ax.set_xlabel("Epoch"); ax.set_ylabel("Loss / Reward")

plt.suptitle("PopQA — Training: Phase 1 (CE+BERTTune) & Phase 2 (GRPO)", fontsize=13)
plt.tight_layout()
plt.savefig(os.path.join(PLOTS_DIR, "training_curves.png"), dpi=150, bbox_inches="tight")
plt.show()

## Cell 16: Evaluate — Latent-RAG (Injection)

Greedy generation on the test split using the GRPO-tuned adapter. Results saved as CSV.

In [ ]:
# ── Evaluate: Latent-RAG Injection ────────────────────────────────────────────
EVAL_DIR = os.path.join(cfg.base_dir, "results")
os.makedirs(EVAL_DIR, exist_ok=True)

INJECTION_CSV = os.path.join(EVAL_DIR, f"injection_test_L{cfg.injection_layer}.csv")


def evaluate_injection(cfg, model, tokenizer, adapter, data, concept_vecs,
                       out_path: str, max_new_tokens: int = 64):
    adapter.eval()
    layer   = model.model.layers[cfg.injection_layer]
    results = []

    for item in tqdm(data, desc="Injection eval"):
        vec = concept_vecs.get(item["entity_key"])
        if vec is None:
            generated = ""
        else:
            msgs       = [{"role": "user", "content": item["question"]}]
            prompt_str = tokenizer.apply_chat_template(msgs, tokenize=False,
                                                       add_generation_prompt=True)
            inputs = tokenizer(prompt_str, return_tensors="pt").to(model.device)

            with torch.no_grad():
                injection_vec = adapter(
                    vec.unsqueeze(0).to(model.device, dtype=cfg.torch_dtype)
                ).squeeze(0)

            inject_idx = inputs["input_ids"].shape[1] - 1

            def _hook(module, args, output):
                hs = output[0] if isinstance(output, tuple) else output
                if inject_idx < hs.shape[1]:
                    hs[0, inject_idx, :] = hs[0, inject_idx, :] + injection_vec
                return (hs,) + output[1:] if isinstance(output, tuple) else hs

            handle = layer.register_forward_hook(_hook)
            try:
                with torch.no_grad():
                    out_ids = model.generate(
                        **inputs,
                        max_new_tokens=max_new_tokens,
                        do_sample=False,
                        pad_token_id=tokenizer.eos_token_id,
                    )
            finally:
                handle.remove()
            generated = tokenizer.decode(
                out_ids[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True
            )

        results.append({
            "category":  item["category_key"],
            "word":      item["entity_original"],
            "question":  item["question"],
            "generated": generated,
            "expected":  item["answer"],
        })

    pd.DataFrame(results).to_csv(out_path, index=False)
    print(f"Saved {len(results)} injection results → {out_path}")
    return results


# Reload GRPO adapter (restart-safe)
adapter = build_adapter(cfg, model).to(model.device, dtype=cfg.torch_dtype)
adapter.load_state_dict(torch.load(cfg.adapter_save_path_grpo, map_location=model.device))

injection_results = evaluate_injection(
    cfg, model, tokenizer, adapter, test_data, concept_vecs_test, INJECTION_CSV
)

## Cell 17: Perfect RAG Baseline

The gold answer (`answers_list[0]`) is placed directly in the context window.
This is the upper bound: the model receives the exact answer it needs to paraphrase.
Fair comparison to Latent-RAG because both use only the answer entity — no extra chunk of text.

In [ ]:
# ── Perfect RAG Baseline ──────────────────────────────────────────────────────
RAG_CSV = os.path.join(EVAL_DIR, "perfect_rag_test.csv")

RAG_SYSTEM = (
    "You are a helpful assistant. Use ONLY the provided context to answer "
    "the question. If the context contains the answer, provide it directly."
)


@torch.inference_mode()
def evaluate_perfect_rag(data: List[dict], out_path: str,
                         max_new_tokens: int = 64) -> List[dict]:
    results = []
    for item in tqdm(data, desc="Perfect RAG"):
        context  = item["answer"]   # gold answer as context
        user_msg = f"Context: {context}\n\nQuestion: {item['question']}"
        msgs = [
            {"role": "system", "content": RAG_SYSTEM},
            {"role": "user",   "content": user_msg},
        ]
        prompt_str = tokenizer.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
        inputs     = tokenizer(prompt_str, return_tensors="pt").to(model.device)
        out_ids    = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id,
        )
        generated = tokenizer.decode(
            out_ids[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True
        )
        results.append({
            "category":  item["category_key"],
            "word":      item["entity_original"],
            "question":  item["question"],
            "generated": generated,
            "expected":  item["answer"],
        })

    pd.DataFrame(results).to_csv(out_path, index=False)
    print(f"Saved {len(results)} Perfect RAG results → {out_path}")
    return results


rag_results = evaluate_perfect_rag(test_data, RAG_CSV)

## Cell 18: Gemini Judge Evaluation

Run Gemini 2.5 Flash as the authoritative judge on both conditions. Results cached.

Metrics:
- **Contains Entity (CE)**: does the generated answer explicitly mention `obj`?
- **Matches Reference (MR)**: does the generated answer convey the same core information as `answers_list[0]`?

In [ ]:
# ── Gemini Judge ──────────────────────────────────────────────────────────────

class EvaluationResult(TypedDict):
    contains_entity:               bool
    contains_entity_explanation:   str
    matches_reference:             bool
    matches_reference_explanation: str


genai.configure(api_key=GEMINI_API_KEY)
gemini_judge = genai.GenerativeModel("gemini-2.5-flash")

EVALUATION_PROMPT = (
    "You are evaluating answers from a question-answering system.\n\n"
    "**Context:**\n"
    "- Entity: {word}\n"
    "- Question: {question}\n"
    "- Generated Answer: {generated}\n"
    "- Reference Answer: {expected}\n\n"
    "**Your Task:**\n"
    "Evaluate the generated answer on two dimensions:\n\n"
    "1. **Contains Entity**: Does the generated answer explicitly mention "
    "or reference the entity \"{word}\"?\n\n"
    "2. **Matches Reference**: Does the generated answer convey the same "
    "core information as the reference answer?\n"
    "   - They don't need to be word-for-word identical.\n"
    "   - A short answer (even just the entity name) is correct if it "
    "provides the key information.\n"
    "   - Focus on whether the generated answer would be considered correct."
)


def evaluate_single_row(row: dict) -> dict:
    prompt = EVALUATION_PROMPT.format(**row)
    try:
        response = gemini_judge.generate_content(
            prompt,
            generation_config=genai.GenerationConfig(
                response_mime_type="application/json",
                response_schema=EvaluationResult,
            ),
        )
        raw = response.text.strip()
        for strip in ["```json", "```"]:
            if raw.startswith(strip): raw = raw[len(strip):]
        if raw.endswith("```"): raw = raw[:-3]
        result = json.loads(raw.strip())
        return {
            "contains_entity":               result.get("contains_entity", False),
            "contains_entity_explanation":   result.get("contains_entity_explanation", ""),
            "matches_reference":             result.get("matches_reference", False),
            "matches_reference_explanation": result.get("matches_reference_explanation", ""),
        }
    except Exception as e:
        return {"contains_entity": False, "contains_entity_explanation": f"Error: {e}",
                "matches_reference": False, "matches_reference_explanation": f"Error: {e}"}


def run_gemini_judge(csv_path: str) -> pd.DataFrame:
    out_path = csv_path.replace(".csv", "_evaluated.csv")
    if os.path.exists(out_path):
        df_check = pd.read_csv(out_path)
        if "contains_entity" in df_check.columns:
            print(f"  Already evaluated: {out_path}")
            return df_check
    df = pd.read_csv(csv_path)
    evals = []
    for _, row in tqdm(df.iterrows(), total=len(df), desc=f"  Judge: {os.path.basename(csv_path)}"):
        evals.append(evaluate_single_row(row.to_dict()))
        time.sleep(0.4)
    for key in evals[0]:
        df[key] = [e[key] for e in evals]
    df.to_csv(out_path, index=False)
    print(f"  Saved: {out_path}")
    return df


print("Running Gemini judge on both conditions...")
df_injection_eval = run_gemini_judge(INJECTION_CSV)
df_rag_eval       = run_gemini_judge(RAG_CSV)

## Cell 19: Results — Comparison Table & Visualization

In [ ]:
# ── Results Comparison ────────────────────────────────────────────────────────

def get_scores(df: pd.DataFrame):
    ce = 100 * pd.to_numeric(df["contains_entity"],  errors="coerce").mean()
    mr = 100 * pd.to_numeric(df["matches_reference"], errors="coerce").mean()
    return ce, mr


ce_inj, mr_inj = get_scores(df_injection_eval)
ce_rag, mr_rag = get_scores(df_rag_eval)

print(f"{'='*58}")
print(f"{'POPQA TEST SET RESULTS':^58}")
print(f"{'='*58}")
print(f"{'Method':<22}  {'Contains Entity':>16}  {'Matches Ref':>14}")
print("-" * 58)
print(f"{'Perfect RAG':<22}  {ce_rag:>15.1f}%  {mr_rag:>13.1f}%")
print(f"{'Latent-RAG (ours)':<22}  {ce_inj:>15.1f}%  {mr_inj:>13.1f}%")
print(f"{'='*58}")

# ── Overall bar chart ─────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(7, 5))
x      = np.arange(2)
width  = 0.3
labels = ["Contains Entity", "Matches Reference"]

for i, (name, scores, color) in enumerate([
    ("Perfect RAG",       [ce_rag, mr_rag], "#FFA726"),
    ("Latent-RAG (ours)", [ce_inj, mr_inj], "#66BB6A"),
]):
    offset = (i - 0.5) * width
    bars   = ax.bar(x + offset, scores, width, label=name, color=color, alpha=0.9)
    for bar in bars:
        ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 1,
                f"{bar.get_height():.1f}%", ha="center", va="bottom", fontsize=10)

ax.set_xticks(x)
ax.set_xticklabels(labels)
ax.set_ylim(0, 115)
ax.set_ylabel("Percentage (%)")
ax.set_title("PopQA Test Set: Latent-RAG vs Perfect RAG")
ax.legend()
ax.yaxis.grid(True, linestyle="--", alpha=0.7)
ax.set_axisbelow(True)
plt.tight_layout()
plt.savefig(os.path.join(PLOTS_DIR, "overall_comparison.png"), dpi=150, bbox_inches="tight")
plt.show()

# ── Per-category breakdown ────────────────────────────────────────────────────
for df, label, color in [
    (df_rag_eval,       "Perfect RAG",       "#FFA726"),
    (df_injection_eval, "Latent-RAG (ours)", "#66BB6A"),
]:
    if "category" not in df.columns and "prop" in df.columns:
        df = df.rename(columns={"prop": "category"})
    cats = sorted(df["category"].str.lower().unique())
    ce_vals = [100 * df[df["category"].str.lower() == c]["contains_entity"].mean()  for c in cats]
    mr_vals = [100 * df[df["category"].str.lower() == c]["matches_reference"].mean() for c in cats]
    x2  = np.arange(len(cats))
    w2  = 0.35
    fig2, ax2 = plt.subplots(figsize=(max(12, len(cats) * 1.1), 5))
    ax2.bar(x2 - w2/2, ce_vals, w2, label="Contains Entity",   color="#4CAF50", alpha=0.85)
    ax2.bar(x2 + w2/2, mr_vals, w2, label="Matches Reference",  color="#2196F3", alpha=0.85)
    ax2.set_xticks(x2); ax2.set_xticklabels(cats, rotation=45, ha="right")
    ax2.set_ylim(0, 115); ax2.set_ylabel("%"); ax2.legend()
    ax2.set_title(f"Per-Category: {label}")
    ax2.yaxis.grid(True, linestyle="--", alpha=0.7); ax2.set_axisbelow(True)
    plt.tight_layout()
    fname = f"per_category_{label.replace(' ', '_').replace('(', '').replace(')', '')}.png"
    plt.savefig(os.path.join(PLOTS_DIR, fname), dpi=150, bbox_inches="tight")
    plt.show()
    plt.close()

print("\nAll plots saved to:", PLOTS_DIR)